In [72]:
import requests
import pandas as pd

## Получим данные о рекламе

In [73]:
ads_table = pd.read_csv('data/ads.csv')

## Загрузим данные по конверсиям

In [74]:
conversion = pd.read_json('conversion.json')

In [75]:
conversion['date_group'] = pd.to_datetime(
    conversion['date_group'],
    unit='ms'
)

## Преобразуем даты

In [76]:
ads_table['date'] = pd.to_datetime(ads_table['date'])

In [77]:
ads_table['date_group'] = ads_table['date'].dt.date
conversion['date_group'] = conversion['date_group'].dt.date

## Сгруппируем данные по конверсиям по дате

In [78]:
conversion_grouped = (
    conversion
    .groupby(['date_group'])
    .agg(visits = ('visits', 'sum'),
        registrations=('registrations', 'sum'))
    .reset_index()
)

## Сгруппируем данные по рекламе

In [79]:
ads_grouped = (
    ads_table
    .groupby(['date_group', 'utm_campaign'])
    .agg(cost = ('cost', 'sum'))
    .reset_index()
)

## Объединим таблицы

In [83]:
ads = conversion_grouped.merge (
    ads_grouped,
    on = 'date_group',
    how = 'left')
ads = ads[
    ['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']
]  

## Заполним пропуски

In [84]:
ads['cost'] = ads['cost'].fillna(0)
ads['utm_campaign'] = ads['utm_campaign'].fillna('none')

## Отсортируем данные по дате

In [89]:
ads = ads.sort_values('date_group')

## Сохраним файл

In [92]:
ads.to_json('./ads.json')